# Quantization from Scratch: Scale, Zero-Point, GPTQ and AWQ

The full code of [Quantization from Scratch: Scale, Zero-Point, GPTQ and AWQ](https://residual-stream.lvtan71.workers.dev/posts/quantization-from-scratch/), run from [its folder](https://github.com/lvtan71/residual-stream-code/tree/main/posts/quantization-from-scratch) in the
order the folder's README gives. Each script is followed by the output it
printed for the post, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/quantization-from-scratch.ipynb)

**Hardware:** Runs on a CPU; Colab's free runtime is enough. About 5 min end to end on the RTX 3090 machine that saved the outputs below.
Downloads Qwen2.5-0.5B (about 1 GB).

The post's numbers come from an Apple M4 laptop. Other hardware and library
versions change timings and the last digits of some floating-point results.
Sampled text and small training runs can move further; where that changes a
number the post discusses, the note above says so.

**Outputs saved from a run on NVIDIA GeForce RTX 3090 on 2026-10-18** (torch 2.14.1+cu130, 5 min; every cell ran). Run the notebook to replace them with your own.

In [1]:
%pip install -q "numpy==2.*" "torch==2.14.*" "transformers==5.17.*" huggingface_hub

# Setup: fetch the companion repo and work in this post's folder.
import os, subprocess, sys
SLUG = "quantization-from-scratch"
ROOT = os.path.abspath("..") if os.path.isdir("../posts") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
os.chdir(os.path.join(ROOT, "posts", SLUG))
print("working in", os.getcwd())

Note: you may need to restart the kernel to use updated packages.
working in residual-stream-code/posts/quantization-from-scratch


## `capture.py`

Runs Qwen2.5-0.5B (base) over the blog's text and records the inputs and weight of the MLP down-projection in layers 12 and 22: `X_<L>.npy` (4,096 tokens), `W_<L>.npy`, and from 32,768 later tokens only GPTQ's `H_big_<L>.npy` and AWQ's `mag_big_<L>.npy`. Not shown in the post, and it has no `.out`: the post's inputs were captured before the post's code was saved. `capture.py` is a reconstruction: it follows the logic of the original capture scripts, which covered layer 12 only and read the model and text from local paths.

In [2]:
!{sys.executable} capture.py

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (163711 > 131072). Running this sequence through the model will result in indexing errors


layer 12: inputs 4096 x 4864, calibration tokens 32768


layer 22: inputs 4096 x 4864, calibration tokens 32768


## `measure.py`

Round-to-nearest (per tensor, per channel, per group of 128; asymmetric, symmetric, NF4), GPTQ and AWQ (scaling, then clipping), implemented in numpy and measured as the layer's output error on 2,048 held-out tokens; and GPTQ with too little calibration.

In [3]:
!{sys.executable} measure.py

worked, 8 weights to 4 bits: scale 0.0520, zero-point 6
  integers [0, 4, 6, 7, 8, 9, 10, 15]
  back     [-0.312, -0.104, 0.0, 0.052, 0.104, 0.156, 0.208, 0.468]



=== layer 12: input channels' mean |x|: median 0.067, largest 0.52 (8x); top 1% of channels carry 3% of the total
round-to-nearest, output error (held-out tokens):
                   8 bits       4 bits       3 bits


per tensor          4.95%       72.79%       94.82%


per channel         1.08%       18.17%       38.32%


group 128           0.64%       10.98%       23.31%


...

group 128           0.63%       10.64%       22.62%


4 bits, blocks of 64: asymmetric 9.40%, symmetric 11.53%, NF4 8.51%
group 128, output error (held-out tokens):


  4 bits: round-to-nearest 10.64%; GPTQ 6.87%; AWQ scaling only 8.15% (alpha 0.4), with clipping 7.70%


  3 bits: round-to-nearest 22.62%; GPTQ 15.06%; AWQ scaling only 17.13% (alpha 0.4), with clipping 15.47%


  GPTQ at 4 bits with 2,048 calibration tokens, by damping (fraction of H's mean diagonal): 0.01: 8.23%, 0.1: 7.03%, 1: 6.95%; on its own calibration tokens (0.01): 2.64%


The post printed:

```text
worked, 8 weights to 4 bits: scale 0.0520, zero-point 6
  integers [0, 4, 6, 7, 8, 9, 10, 15]
  back     [-0.312, -0.104, 0.0, 0.052, 0.104, 0.156, 0.208, 0.468]

=== layer 12: input channels' mean |x|: median 0.067, largest 0.52 (8x); top 1% of channels carry 3% of the total
round-to-nearest, output error (held-out tokens):
                   8 bits       4 bits       3 bits
per tensor          4.95%       72.79%       94.82%
per channel         1.08%       18.17%       38.32%
group 128           0.64%       10.98%       23.31%
4 bits, blocks of 64: asymmetric 9.66%, symmetric 11.74%, NF4 9.34%
group 128, output error (held-out tokens):
  4 bits: round-to-nearest 10.98%; GPTQ 8.99%; AWQ scaling only 10.90% (alpha 0.2), with clipping 10.37%
  3 bits: round-to-nearest 23.31%; GPTQ 19.44%; AWQ scaling only 23.25% (alpha 0.2), with clipping 20.52%
  GPTQ at 4 bits with 2,048 calibration tokens, by damping (fraction of H's mean diagonal): 0.01: 12.78%, 0.1: 10.48%, 1: 9.46%; on its own calibration tokens (0.01): 3.76%

=== layer 22: input channels' mean |x|: median 0.146, largest 8.53 (58x); top 1% of channels carry 12% of the total
round-to-nearest, output error (held-out tokens):
                   8 bits       4 bits       3 bits
per tensor          3.69%       57.74%       85.07%
per channel         1.03%       17.38%       36.14%
group 128           0.63%       10.64%       22.62%
4 bits, blocks of 64: asymmetric 9.40%, symmetric 11.53%, NF4 8.51%
group 128, output error (held-out tokens):
  4 bits: round-to-nearest 10.64%; GPTQ 6.87%; AWQ scaling only 8.15% (alpha 0.4), with clipping 7.70%
  3 bits: round-to-nearest 22.62%; GPTQ 15.06%; AWQ scaling only 17.13% (alpha 0.4), with clipping 15.47%
  GPTQ at 4 bits with 2,048 calibration tokens, by damping (fraction of H's mean diagonal): 0.01: 8.23%, 0.1: 7.03%, 1: 6.95%; on its own calibration tokens (0.01): 2.64%
```

## `decode_speed.py`

A roofline bound on one decode step of an 8B model on an H100, BF16 against INT4 weights, by batch size.

In [4]:
!{sys.executable} decode_speed.py

 batch  BF16 step  INT4 step  speedup   bound by
     1     4.78 ms     1.27 ms     3.8x   BF16 memory, INT4 memory
     8     4.78 ms     1.27 ms     3.8x   BF16 memory, INT4 memory
    64     4.78 ms     1.27 ms     3.8x   BF16 memory, INT4 memory
   128     4.78 ms     2.07 ms     2.3x   BF16 memory, INT4 compute
   256     4.78 ms     4.14 ms     1.2x   BF16 memory, INT4 compute
  1024    16.57 ms    16.57 ms     1.0x   BF16 compute, INT4 compute
compute-bound from a batch of 78 (INT4) and 295 (BF16)


The post printed:

```text
 batch  BF16 step  INT4 step  speedup   bound by
     1     4.78 ms     1.27 ms     3.8x   BF16 memory, INT4 memory
     8     4.78 ms     1.27 ms     3.8x   BF16 memory, INT4 memory
    64     4.78 ms     1.27 ms     3.8x   BF16 memory, INT4 memory
   128     4.78 ms     2.07 ms     2.3x   BF16 memory, INT4 compute
   256     4.78 ms     4.14 ms     1.2x   BF16 memory, INT4 compute
  1024    16.57 ms    16.57 ms     1.0x   BF16 compute, INT4 compute
compute-bound from a batch of 78 (INT4) and 295 (BF16)
```